# Error-Specific LoRA Training for Decompilation

This notebook trains a LoRA adapter on synthetic data with Ghidra artifacts and semantic errors.

## 1. Setup

In [ ]:
import os, re
!pip install uv
if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 48.4 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Resolved 8 packages in 21ms
Prepared 5 packages in 2.51s
Installed 5 packages in 7ms
 + bitsandbytes==0.49.0
 + cut-cross-entropy==25.1.1
 + trl==0.26.2
 + unsloth-zoo==2026.1.2
 + xformers==0.0.33.post1
Using Python 3.12.12 environment at: /usr
Resolved 39 packages in 278ms
Prepared 2 packages in 562ms
Uninstalled 2 packages in 36ms
Installed 2 packages in 14ms
 - datasets==4.0.0
 + datasets==4.3.0
 - pyarrow==18.1.0
 + pyarrow==22.0.0
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 14ms
Prepared 1 package in 21ms
Installed 1 package in 3ms
 + unsloth==2026.1.2
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 103ms
Prepared 1 package in 450ms
Uninstalled 1 package in 450ms
Installed 1 package in 48ms
 - transformers==4.57.3
 + transformers==4.56.2
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 1ms
Prepared 1 package 

## 2. Upload Training Data


In [ ]:
from google.colab import files

print("Upload ghidra_semantic.jsonl:")
uploaded = files.upload()

Upload ghidra_semantic.jsonl:


Saving training_data.jsonl to training_data.jsonl


In [ ]:
import json

# Load training data
def load_jsonl(path):
    samples = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
    return samples

synthetic = load_jsonl('ghidra_semantic.jsonl')
original = load_jsonl("training_data.jsonl")
print(f"Loaded {len(synthetic) + len(original)} training samples")

Loaded 5206 training samples


## 4. Load Model

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "codellama/CodeLlama-7b-Instruct-hf",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

codellama/CodeLlama-7b-Instruct-hf does not have a padding token! Will use pad_token = <unk>.


## 5. Setup LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


Unsloth 2026.1.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 6. Prepare Dataset

In [ ]:
def formatting_prompts_func(examples):
    """Format training samples with CodeLlama instruct template."""
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        text = (
            f"<s>[INST] You are an expert C decompiler.\n"
            f"Refine the following Ghidra pseudocode into valid, compilable C code.\n"
            f"STRICT RESPONSE RULES:\n"
            f"1. Do not write a main function.\n"
            f"2. Keep the exact same function name and arguments.\n"
            f"3. Output ONLY the raw code. Do not use Markdown code blocks (```).\n"
            f"4. Do not output any introductory text or explanations.\n\n"
            f"Pseudocode:\n"
            f"{input_text}\n"
            f"[/INST]\n"
            f"{output_text}</s>"
        )
        texts.append(text)
    return {"text": texts}

# Create dataset
import random
random.seed(3407)
n_original = min(len(synthetic * 2), len(original))
original_sample = random.sample(original, n_original)

training_samples = synthetic + original_sample
random.shuffle(training_samples)

from datasets import Dataset
dataset = Dataset.from_list(training_samples)
dataset = dataset.train_test_split(test_size=0.1, seed=3407)
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/3242 [00:00<?, ? examples/s]

Map:   0%|          | 0/361 [00:00<?, ? examples/s]

## 7. Train

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type = "linear",
        output_dir=f"error_lora_semantic/checkpoints",
        logging_strategy="steps",
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        seed=3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/3242 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/361 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,242 | Num Epochs = 3 | Total steps = 609
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 159,907,840 of 6,898,454,528 (2.32% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.417900,0.324230
2,0.259600,0.307238
3,0.199600,0.310572


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## 8. Save Model

In [ ]:
print(f"Saving LoRA adapter to error_lora_semantic...")
model.save_pretrained("error_lora_semantic")
tokenizer.save_pretrained("error_lora_semantic")
print("Done!")

Saving LoRA adapter to error_lora_semantic...
Done!


## 9. Download Model

In [ ]:
# Zip and download the adapter
!zip -r error_lora_semantic.zip error_lora_semantic/

from google.colab import files
files.download('error_lora_semantic.zip')

  adding: error_lora_semantic/ (stored 0%)
  adding: error_lora_semantic/adapter_config.json (deflated 57%)
  adding: error_lora_semantic/README.md (deflated 65%)
  adding: error_lora_semantic/chat_template.jinja (deflated 61%)
  adding: error_lora_semantic/tokenizer.json (deflated 85%)
  adding: error_lora_semantic/tokenizer_config.json (deflated 77%)
  adding: error_lora_semantic/checkpoints/ (stored 0%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/ (stored 0%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/adapter_config.json (deflated 57%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/trainer_state.json (deflated 81%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/README.md (deflated 65%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/scheduler.pt (deflated 61%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/training_args.bin (deflated 53%)
  adding: error_lora_semantic/checkpoints/checkpoint-406/chat_template.jinja (de

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>